In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Cancer Data").getOrCreate()
sc = spark.sparkContext

In [3]:
import pyspark
print(pyspark.__version__)

3.5.0


In [ ]:
df = spark.read.csv("Lung Cancer Dataset.csv", header=True, inferSchema=True)

In [44]:
df.show(5)

+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  1|64.0|  Male|     Sweden|    2016-04-05|     Stage I|           Yes|Passive Smoker|29.4|              199|           0|     0|        1|           0|  Chemotherapy|        2017-09-10|       0|
|  2|50.0|Female|Netherlands|    2023-04-20|   Stage III|           Yes|Passive Smoker|41.2|              280|           1|     1|        0|           0|       Surgery|        2024-06-17|       1|
|  3|65.0|Femal

In [5]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: string (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)



In [ ]:
#Function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format.  
from pyspark.sql.functions import col, when
from pyspark.sql.types import IntegerType, DoubleType, DateType, StringType

def clean_cancer_data(df):
    #Removing duplicates
    df = df.dropDuplicates()
    #Converting datatypes
    schema_map = {
        "id": IntegerType(),
        "age": DoubleType(),
        "gender": StringType(),
        "country": StringType(),
        "diagnosis_date": DateType(),
        "cancer_stage": StringType(),
        "family_history": StringType(),   # will convert later
        "smoking_status": StringType(),
        "bmi": DoubleType(),
        "cholesterol_level": IntegerType(),
        "hypertension": IntegerType(),
        "asthma": IntegerType(),
        "cirrhosis": IntegerType(),
        "other_cancer": IntegerType(),
        "treatment_type": StringType(),
        "end_treatment_date": DateType(),
        "survived": IntegerType()
    }

    #Applying casting to all columns
    for col_name, dtype in schema_map.items():
        df = df.withColumn(col_name, col(col_name).cast(dtype))
    #Converting Yes/No columns to 1/0
    yes_no_cols = ["family_history"]
    for c in yes_no_cols:
        df = df.withColumn(
            c,
            when(col(c) == "Yes",1)
            .when(col(c) == "No", 0)
            .otherwise(None)
        )
    return df

In [27]:
df_cleaned = clean_cancer_data(df)

In [18]:
df_cleaned.printSchema()

root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: integer (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)



In [28]:
df_cleaned.show(2)

+---+----+------+-------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+-------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|220|58.0|  Male|Croatia|    2020-12-25|   Stage III|             1|Current Smoker|41.6|              252|           1|     0|        0|           0|     Radiation|        2021-12-08|       0|
|259|57.0|  Male|Germany|    2015-02-23|   Stage III|             0| Former Smoker|42.9|              280|           1|     0|        0|           0|       Surgery|        2016-01-02|       0|
+---+----+------+-------+----------

In [ ]:
#Function that adds a new column, treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. 
#Then, return the average treatment duration for each treatment type.  


from pyspark.sql.functions import datediff, avg

def treatment_duration_analysis(df):
    df = df.withColumn(
        "treatment_duration_days",
        datediff(col("end_treatment_date"), col("diagnosis_date"))
    )
    result = df.groupBy("treatment_type").agg(avg("treatment_duration_days").alias("avg_duration_days"))
    return result

In [31]:
avg_treatment_duration_by_type = treatment_duration_analysis(df_cleaned)

In [32]:
avg_treatment_duration_by_type.show()

+--------------+------------------+
|treatment_type| avg_duration_days|
+--------------+------------------+
|     Radiation|458.40320462900917|
|  Chemotherapy|458.39540091909953|
|      Combined| 457.8152186120058|
|       Surgery|457.73744630723684|
+--------------+------------------+



In [33]:
#Task 3: Write a function that returns the smoking_status group with the highest survival rate.  
def highest_survival_by_smoking(df):
    survival_rates = df.groupBy("smoking_status").agg(avg("survived").alias("survival_rate"))
    result = survival_rates.orderBy(col("survival_rate").desc()).limit(1)
    return result

In [34]:
highest_survival_rate = highest_survival_by_smoking(df_cleaned)

In [37]:
highest_survival_rate.show()

+--------------+-------------------+
|smoking_status|      survival_rate|
+--------------+-------------------+
|  Never Smoked|0.22091034383684025|
+--------------+-------------------+



In [ ]:
#function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV
from pyspark.sql.functions import count, sum
def top_stage4_countries(df):
    country_stats = df.groupBy("country").agg(
        count("*").alias("total_patients"),
        sum(when(col("cancer_stage") == "Stage IV", 1).otherwise(0)).alias("stage4_count")
    )
    result = country_stats.withColumn(
    "stage4_percentage",
    (col("stage4_count")/col("total_patients"))*100
    ).orderBy(col("stage4_percentage").desc()).limit(3)
    return result

In [39]:
top_stage_4_countries = top_stage4_countries(df_cleaned)

In [40]:
top_stage_4_countries.show()

+--------------+--------------+------------+------------------+
|       country|total_patients|stage4_count| stage4_percentage|
+--------------+--------------+------------+------------------+
|        Greece|         33052|        8429| 25.50223889628464|
|       Croatia|         33138|        8426|25.427002233085883|
|Czech Republic|         32885|        8317|25.291166185190818|
+--------------+--------------+------------+------------------+



In [ ]:
#Function that filters patients who:  
#Are male  
#Diagnosed in Stage III or IV  
#Have a family history of cancer  
#Are current smokers  
#Have a BMI > 30  
#Survived 
#Returning the average age and the percentage of these patients who had hypertension.
def high_risk_patient_analysis(df):
    filtered_df = df.filter(
        (col("gender") == "Male") &
        (col("cancer_stage").isin("Stage III", "Stage IV")) &
        (col("family_history") == 1) &
        (col("smoking_status") == "Current Smoker") &
        (col("bmi") > 30) &
        (col("survived") == 1)
    )
    result = filtered_df.agg(
        avg("age").alias("average_age"),(avg("hypertension")*100).alias("hypertension_percentage"))
    return result
    

In [42]:
hypertension_percentage_df = high_risk_patient_analysis(df_cleaned)

In [43]:
hypertension_percentage_df.show()

+------------------+-----------------------+
|       average_age|hypertension_percentage|
+------------------+-----------------------+
|55.179398872886665|      74.76518472135254|
+------------------+-----------------------+



Assumptions:

- survived is assumed to be:
    1 = survived
    0 = not survived
- avg(survived) = survival rate
- avg(hypertension) works because it's binary (0/1)